In [1]:
import duckdb
import pandas as pd

# Configuration immédiate
duckdb.execute("SET force_download=true")
duckdb.execute("SET threads TO 4")  # Utilise plusieurs threads pour accélérer


In [2]:
# Supprimer les anciennes vues
duckdb.execute("DROP VIEW IF EXISTS titles")
duckdb.execute("DROP VIEW IF EXISTS ratings")
duckdb.execute("DROP VIEW IF EXISTS actors")
duckdb.execute("DROP VIEW IF EXISTS name")
duckdb.execute("DROP VIEW IF EXISTS crew")
duckdb.execute("DROP VIEW IF EXISTS akas")

In [3]:
url_titles = 'https://datasets.imdbws.com/title.basics.tsv.gz'
url_ratings = 'https://datasets.imdbws.com/title.ratings.tsv.gz'
url_actors = 'https://datasets.imdbws.com/title.principals.tsv.gz'
url_akas = "https://datasets.imdbws.com/title.akas.tsv.gz"
url_tmdb = 'D:/Utilisateurs/Dylan Malis/Téléchargements/tmdb_full.csv'
url_crew = "https://datasets.imdbws.com/title.crew.tsv.gz"
url_name = "https://datasets.imdbws.com/name.basics.tsv.gz"

In [4]:
duckdb.query(f"CREATE OR REPLACE VIEW titles AS SELECT * FROM read_csv_auto('{url_titles}')")
duckdb.query(f"CREATE OR REPLACE VIEW tmdb AS SELECT * FROM read_csv_auto('{url_tmdb}')")
duckdb.query(f"CREATE OR REPLACE VIEW akas AS SELECT * FROM read_csv_auto('{url_akas}')")
duckdb.query(f"CREATE OR REPLACE VIEW crew AS SELECT * FROM read_csv_auto('{url_crew}')")
duckdb.query(f"CREATE OR REPLACE VIEW name AS SELECT * FROM read_csv_auto('{url_name}')")
duckdb.query(f"CREATE OR REPLACE VIEW rating AS SELECT * FROM read_csv_auto('{url_ratings}')")



In [5]:
import pandas as pd
pd.set_option('display.max_rows', 500)

query_test = """
SELECT titles.tconst, tmdb.original_title, titles.primaryTitle, akas.title, akas.attributes, akas.types, akas.ordering,  titles.startYear,titles.runtimeMinutes,akas.language, tmdb.spoken_languages,titles.genres, name.primaryName,name.primaryProfession, name.knownForTitles, rating.averageRating,tmdb.popularity,tmdb.tagline, tmdb.overview,tmdb.poster_path, tmdb.video, tmdb.budget,tmdb.revenue
    FROM titles
    LEFT JOIN tmdb ON imdb_id = titles.tconst
    LEFT JOIN akas ON akas.titleId = titles.tconst
    LEFT JOIN crew ON crew.tconst = titles.tconst
    LEFT JOIN name ON name.nconst = crew.directors
    LEFT JOIN rating ON rating.tconst = titles.tconst
    WHERE titles.tconst = 'tt0119116'
    AND akas.region = 'FR'
    LIMIT 1000
"""

In [6]:
dftest = duckdb.query(query_test).to_df()
dftest.head(100)

,tconst,original_title,primaryTitle,title,attributes,types,ordering,startYear,runtimeMinutes,language,...,primaryProfession,knownForTitles,averageRating,popularity,tagline,overview,poster_path,video,budget,revenue
0,tt0119116,The Fifth Element,The Fifth Element,Le Cinquième Élément,\N,imdbDisplay,35,1997,126,\N,...,"writer,producer,director","tt0119116,tt2872732,tt0110413,tt0100263",7.6,46.823,There is no future without it.,"In 2257, a taxi driver is unintentionally give...",/fPtlCO1yQtnoLHOwKtWz7db6RGU.jpg,False,90000000,263920180


In [9]:
import pandas as pd
pd.set_option('display.max_rows', 500)

query_finale = """
    SELECT titles.tconst, tmdb.original_title, titles.primaryTitle, akas.title, akas.attributes, akas.types, akas.ordering,  titles.startYear,titles.runtimeMinutes,akas.language, tmdb.spoken_languages,titles.genres, name.primaryName,name.primaryProfession, name.knownForTitles, rating.averageRating,tmdb.popularity,tmdb.tagline, tmdb.overview,tmdb.poster_path, tmdb.video, tmdb.budget,tmdb.revenue
    FROM titles
    LEFT JOIN tmdb ON imdb_id = titles.tconst
    LEFT JOIN akas ON akas.titleId = titles.tconst
    LEFT JOIN crew ON crew.tconst = titles.tconst
    LEFT JOIN name ON name.nconst = crew.directors
    LEFT JOIN rating ON rating.tconst = titles.tconst
    WHERE titles.titleType = 'movie'
    AND titles.tconst NOT LIKE '%\\N%'
    AND titles.isAdult = 0
    AND titles.genres IS NOT NULL
    AND titles.genres NOT LIKE '%\\N%'
    AND titles.genres NOT LIKE '%Horror%'
    AND titles.genres NOT LIKE '%NaN%'
    AND titles.runtimeMinutes NOT LIKE '%\\N%'
    AND CAST(titles.runtimeMinutes AS INTEGER) >= 80
    AND CAST(titles.runtimeMinutes AS INTEGER) <= 220
    AND akas.region = 'FR' AND (tmdb.spoken_languages LIKE '%fr%' OR tmdb.spoken_languages LIKE '%en%')
    AND titles.startYear NOT LIKE '%\\N%'
    AND CAST(titles.startYear as INTEGER) BETWEEN 1960 AND 2027
    AND tmdb.status != 'Canceled'
    AND rating.averageRating >= 5.0

    
"""
# on viens de rajouter le filtre rating, mais pas tester.
#akas.region = 'FR' AND
#AND tmdb.spoken_languages LIKE '%fr%'


In [10]:
# EXÉCUTER LA REQUÊTE ET OBTENIR LE RÉSULTAT
df = duckdb.query(query_finale).to_df()
df.head(50)

,tconst,original_title,primaryTitle,title,attributes,types,ordering,startYear,runtimeMinutes,language,...,primaryProfession,knownForTitles,averageRating,popularity,tagline,overview,poster_path,video,budget,revenue
0,tt0052205,The Small World of Sammy Lee,The Small World of Sammy Lee,Les folles nuits de Sammy Lee,\N,imdbDisplay,10,1963,107,\N,...,"writer,director,producer","tt0062803,tt0082812,tt0054403,tt0061452",7.1,1.960,Soho... Where Love Comes Cheap... Money Comes ...,The compère of a seedy strip club struggles to...,/4Trt44jLU4rTRYg7JK8dbVdXNfe.jpg,False,0,0
1,tt0052376,War Is Hell,War Is Hell,Héros de guerre,\N,imdbDisplay,9,1961,81,\N,...,"producer,director,writer","tt0065815,tt0074377,tt0053332,tt0052739",5.3,1.400,IRON-GUTS GUYS IN ACTION!,"During the Korean War, a glory-hunting sergean...",/4QyQdnsLNzfGeN1Gzu6xMa7EUz2.jpg,False,0,0
2,tt0052607,The Battle of the Sexes,The Battle of the Sexes,La bataille des sexes,\N,\N,13,1960,84,\N,...,"director,editor,writer","tt0095159,tt0037635,tt0045201,tt0044829",6.6,3.273,NaN,Angela Barrows is a man-eating business woman ...,/i3b1RowQT7IqUyU99xmA3CaQAOt.jpg,False,0,0
3,tt0052614,Le Bel Âge,Le bel âge,Le bel âge,\N,imdbDisplay,3,1960,100,\N,...,"director,writer,assistant_director","tt0052614,tt0057635,tt0208040,tt0158774",6.8,1.400,NaN,"Steph, Jean-Claude and Jacques work in a Paris...",/98Z5yPoNIm8sQeyep5cpr5NHov9.jpg,False,0,0
4,tt0052680,Cash McCall,Cash McCall,Cet homme est un requin,\N,\N,12,1960,102,\N,...,"director,actor,producer","tt0060028,tt0051051,tt0050681,tt0042953",6.4,3.742,High finance and high romance are about to merge.,Wealthy hotshot Cash McCall makes his money by...,/v0wUowsxFuYVm2VvmIty2lZqUoO.jpg,False,0,0
5,tt0052686,Certains l'aiment... froide,Some Like It... Cold,Certains l'aiment... froide!,\N,imdbDisplay,7,1960,90,\N,...,"assistant_director,director,writer","tt0315099,tt0054394,tt0052686,tt0159605",5.2,1.916,NaN,The old Valmorin died 200 years ago. The notar...,/nWxNWddjRO14xGOiHU770oDfXsV.jpg,False,0,0
6,tt0052698,Classe tous risques,The Big Risk,Classe tous risques,\N,imdbDisplay,8,1960,103,\N,...,"writer,director,assistant_director","tt0105682,tt0113947,tt0064165,tt0075975",7.5,5.843,Tough Gangster Action,"On crowded Milan streets, two men execute a sp...",/uMiuUAlPwiWDKSgu4SoShpzvMR.jpg,False,0,0
7,tt0052745,A Dog of Flanders,A Dog of Flanders,L'Orphelin et son chien,\N,imdbDisplay,13,1960,96,\N,...,"director,editor,miscellaneous","tt0033729,tt0058241,tt0050105,tt0064708",7.2,1.400,Wonderful family entertainment the whole world...,"The emotional story of a boy, his grandfather,...",/sf50zn1FhPxf8tvvlsYTYadkSAv.jpg,False,0,0
8,tt0052769,L'Eau à la bouche,L'eau à la bouche,L'eau à la bouche,\N,imdbDisplay,11,1960,85,\N,...,"director,actor,writer","tt0054838,tt0066041,tt0210108,tt0053725",6.1,1.400,NaN,Miléna is living in her grandmother's baroque ...,/nq4i6WvKAZokanPCz2wXIDqDKc0.jpg,False,0,0
9,tt0052781,Era notte a Roma,Escape by Night,Les Évadés de la nuit,\N,imdbDisplay,29,1960,82,\N,...,"writer,director,producer","tt0038823,tt0038890,tt0046511,tt0039417",7.1,1.782,NaN,"In Nazi-occupied Rome, a beautiful bootlegger,...",/7k40hZ9b5pMIUge74mZgY40yOdL.jpg,False,0,0


In [11]:
df.describe()

,ordering,averageRating,popularity,budget,revenue
count,26195.000000,26195.000000,26195.000000,2.619500e+04,2.619500e+04
mean,13.314449,6.360199,10.531008,8.661224e+06,2.391708e+07
std,10.704109,0.756025,58.617900,2.653033e+07,1.030815e+08
min,2.000000,5.000000,0.600000,0.000000e+00,0.000000e+00
25%,6.000000,5.800000,1.936000,0.000000e+00,0.000000e+00
50%,10.000000,6.300000,5.286000,0.000000e+00,0.000000e+00
75%,18.000000,6.900000,10.559000,2.300000e+06,1.356889e+06
max,150.000000,9.400000,4665.438000,5.793304e+08,2.923706e+09


In [12]:
df.shape

(26195, 23)

In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 26195 entries, 0 to 26194
Data columns (total 23 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   tconst             26195 non-null  str    
 1   original_title     26195 non-null  str    
 2   primaryTitle       26195 non-null  str    
 3   title              26195 non-null  str    
 4   attributes         26195 non-null  str    
 5   types              26195 non-null  str    
 6   ordering           26195 non-null  int64  
 7   startYear          26195 non-null  str    
 8   runtimeMinutes     26195 non-null  str    
 9   language           26195 non-null  str    
 10  spoken_languages   26195 non-null  str    
 11  genres             26195 non-null  str    
 12  primaryName        24245 non-null  str    
 13  primaryProfession  24245 non-null  str    
 14  knownForTitles     24245 non-null  str    
 15  averageRating      26195 non-null  float64
 16  popularity         26195 non-null

In [14]:
df.to_csv(
    "resultats_cinema4.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig"
)